# Toy Lund-plane sensitivity study

This notebook builds a deliberately simple toy model for emissions in the Lund plane and asks which regions help separate boosted $W$ jets from QCD jets, and boosted top jets from QCD jets.

It is not a replacement for a full shower simulation. The point is to make the logic visible: inject a few physically motivated emission patterns, histogram the Lund plane, and inspect which bins carry the strongest discriminating power.

## Coordinates and toy assumptions

We use

- $x = \ln(1/\Delta)$, so smaller $x$ means wider-angle branchings.
- $y = \ln(k_t / \mathrm{GeV})$, so larger $y$ means harder transverse momentum in the splitting.

The toy generator contains a QCD-like soft/collinear background plus optional signal islands:

- $W$ jets receive one hard two-prong splitting around the $W$ mass scale.
- top jets receive a top-scale wide-angle splitting and often a secondary $W$-like splitting.

The numerical locations are chosen for interpretability, not precision.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(7)
plt.rcParams.update({"figure.figsize": (7, 5), "axes.grid": False})

In [ ]:
X_RANGE = (-0.2, 4.5)   # ln(1/Delta)
Y_RANGE = (-1.0, 5.7)   # ln(kT / GeV)
XBINS = np.linspace(*X_RANGE, 30)
YBINS = np.linspace(*Y_RANGE, 32)


def clip_points(points):
    points = np.asarray(points, dtype=float)
    keep = (
        (points[:, 0] >= X_RANGE[0]) & (points[:, 0] <= X_RANGE[1]) &
        (points[:, 1] >= Y_RANGE[0]) & (points[:, 1] <= Y_RANGE[1])
    )
    return points[keep]


def sample_qcd_emissions(n_soft):
    # Broad soft/collinear population: many emissions at lower kT, with a mild
    # trend toward lower kT at more collinear angles.
    x = rng.gamma(shape=2.1, scale=0.95, size=n_soft)
    y_mean = 2.25 - 0.28 * x
    y = rng.normal(y_mean, 0.95, size=n_soft)
    return np.column_stack([x, y])


def gaussian_island(mean, cov, n=1):
    return rng.multivariate_normal(mean, cov, size=n)


def sample_jet(kind):
    n_soft = rng.poisson(18)
    points = [sample_qcd_emissions(n_soft)]

    if kind == "w":
        # A resolved W -> qq splitting: moderately collimated and hard.
        points.append(gaussian_island(mean=[1.55, 3.75], cov=[[0.10, 0.02], [0.02, 0.16]]))
        if rng.random() < 0.35:
            points.append(gaussian_island(mean=[2.25, 2.65], cov=[[0.18, 0.00], [0.00, 0.18]]))

    if kind == "top":
        # Top -> bW: a harder and wider-angle splitting than the W island.
        points.append(gaussian_island(mean=[0.85, 4.45], cov=[[0.12, -0.01], [-0.01, 0.18]]))
        # The hadronic W inside the top remains visible, but with more spread.
        if rng.random() < 0.85:
            points.append(gaussian_island(mean=[1.75, 3.55], cov=[[0.16, 0.03], [0.03, 0.20]]))
        if rng.random() < 0.45:
            points.append(gaussian_island(mean=[2.55, 2.55], cov=[[0.20, 0.00], [0.00, 0.24]]))

    return clip_points(np.vstack(points))


def sample_dataset(kind, n_jets):
    return [sample_jet(kind) for _ in range(n_jets)]

## Generate toy samples

In [ ]:
N_JETS = 6000
qcd_jets = sample_dataset("qcd", N_JETS)
w_jets = sample_dataset("w", N_JETS)
top_jets = sample_dataset("top", N_JETS)

print(f"QCD jets: {len(qcd_jets)}")
print(f"W jets:   {len(w_jets)}")
print(f"top jets: {len(top_jets)}")

In [ ]:
def plot_emission_cloud(jets, title, ax):
    points = np.vstack(jets[:350])
    ax.scatter(points[:, 0], points[:, 1], s=4, alpha=0.18, linewidths=0)
    ax.set_title(title)
    ax.set_xlabel(r"$\ln(1/\Delta)$")
    ax.set_ylabel(r"$\ln(k_t / \mathrm{GeV})$")
    ax.set_xlim(X_RANGE)
    ax.set_ylim(Y_RANGE)


fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True)
plot_emission_cloud(qcd_jets, "QCD-like background", axes[0])
plot_emission_cloud(w_jets, "Toy W jets", axes[1])
plot_emission_cloud(top_jets, "Toy top jets", axes[2])
fig.tight_layout()

## Histogram each jet into Lund-plane bins

Each jet becomes a 2D count image. We use two complementary sensitivity measures:

- occupancy difference: signal mean count minus QCD mean count per bin;
- logistic-regression coefficient: a simple multivariate tagger showing bins that help classify the signal.

In [ ]:
def jet_histogram(jet):
    hist, _, _ = np.histogram2d(jet[:, 0], jet[:, 1], bins=[XBINS, YBINS])
    return hist


def histogram_sample(jets):
    return np.stack([jet_histogram(jet) for jet in jets])


H_qcd = histogram_sample(qcd_jets)
H_w = histogram_sample(w_jets)
H_top = histogram_sample(top_jets)

print("Histogram shape per sample:", H_qcd.shape)

In [ ]:
def fit_linear_tagger(signal_hists, background_hists):
    X = np.concatenate([background_hists, signal_hists]).reshape(2 * len(signal_hists), -1)
    y = np.concatenate([np.zeros(len(background_hists)), np.ones(len(signal_hists))])

    # Log-compress counts so one busy soft region does not dominate only by multiplicity.
    X = np.log1p(X)
    clf = LogisticRegression(max_iter=1500, C=0.35, solver="lbfgs")
    clf.fit(X, y)
    score = clf.decision_function(X)
    auc = roc_auc_score(y, score)
    coef = clf.coef_[0].reshape(len(XBINS) - 1, len(YBINS) - 1)
    return clf, coef, auc


w_clf, w_coef, w_auc = fit_linear_tagger(H_w, H_qcd)
top_clf, top_coef, top_auc = fit_linear_tagger(H_top, H_qcd)

print(f"Toy W-vs-QCD AUC:   {w_auc:.3f}")
print(f"Toy top-vs-QCD AUC: {top_auc:.3f}")

In [ ]:
def plot_map(values, title, ax, cmap="coolwarm", symmetric=True):
    if symmetric:
        vmax = np.max(np.abs(values))
        vmin = -vmax
    else:
        vmin, vmax = None, None
    mesh = ax.pcolormesh(XBINS, YBINS, values.T, cmap=cmap, shading="auto", vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel(r"$\ln(1/\Delta)$")
    ax.set_ylabel(r"$\ln(k_t / \mathrm{GeV})$")
    return mesh


w_delta = H_w.mean(axis=0) - H_qcd.mean(axis=0)
top_delta = H_top.mean(axis=0) - H_qcd.mean(axis=0)

fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True, sharey=True)
m = plot_map(w_delta, "W - QCD occupancy", axes[0, 0])
fig.colorbar(m, ax=axes[0, 0], label="mean count difference")
m = plot_map(top_delta, "top - QCD occupancy", axes[0, 1])
fig.colorbar(m, ax=axes[0, 1], label="mean count difference")
m = plot_map(w_coef, "linear tagger coefficients: W", axes[1, 0])
fig.colorbar(m, ax=axes[1, 0], label="coefficient")
m = plot_map(top_coef, "linear tagger coefficients: top", axes[1, 1])
fig.colorbar(m, ax=axes[1, 1], label="coefficient")
fig.tight_layout()

## Rank the most sensitive Lund-plane regions

The table below reports the bins with the largest positive linear coefficients. Positive means the bin makes the simple classifier more signal-like. Negative bins, not shown here, are QCD-like veto regions.

In [ ]:
def top_bins(coef, n=10):
    flat = coef.ravel()
    order = np.argsort(flat)[::-1][:n]
    rows = []
    for idx in order:
        ix, iy = np.unravel_index(idx, coef.shape)
        rows.append({
            "x_center": 0.5 * (XBINS[ix] + XBINS[ix + 1]),
            "y_center": 0.5 * (YBINS[iy] + YBINS[iy + 1]),
            "coef": flat[idx],
        })
    return rows


def print_bins(title, rows):
    print(title)
    print("rank   ln(1/Delta)   ln(kT/GeV)   coefficient")
    for i, row in enumerate(rows, start=1):
        print(f"{i:>4}   {row['x_center']:>11.2f}   {row['y_center']:>10.2f}   {row['coef']:>11.3f}")
    print()


print_bins("Most W-sensitive bins", top_bins(w_coef, 10))
print_bins("Most top-sensitive bins", top_bins(top_coef, 10))

In [ ]:
shared = np.minimum(np.maximum(w_coef, 0), np.maximum(top_coef, 0))
top_unique = np.maximum(top_coef, 0) - shared
w_unique = np.maximum(w_coef, 0) - shared

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True)
m = plot_map(w_unique, "more W-specific", axes[0], cmap="viridis", symmetric=False)
fig.colorbar(m, ax=axes[0], label="positive coefficient excess")
m = plot_map(shared, "shared W/top sensitivity", axes[1], cmap="viridis", symmetric=False)
fig.colorbar(m, ax=axes[1], label="shared positive coefficient")
m = plot_map(top_unique, "more top-specific", axes[2], cmap="viridis", symmetric=False)
fig.colorbar(m, ax=axes[2], label="positive coefficient excess")
fig.tight_layout()

## Toy-study interpretation

In this construction, $W$ tagging is most sensitive to a hard, moderately collimated two-prong island: intermediate $\ln(1/\Delta)$ and high $\ln(k_t)$.

Top tagging inherits some of that same $W$-like sensitivity, but it also gains a more top-specific contribution from wider-angle and harder branchings. That is the toy analogue of resolving the top decay structure beyond a single $W \to q\bar q'$ splitting.

For a real study, the same workflow can be applied to simulated or experimental Lund declusterings: build per-jet Lund histograms, train a sparse/simple classifier, and compare coefficient maps or leave-one-region-out performance for top-vs-QCD and W-vs-QCD.